In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import altair as alt
from sklearn.manifold import MDS
from sklearn.cluster import KMeans

## Judge Analysis: MDS & Clustering

Prototype judge map using federal supreme court justice voting records from [Washington University Law](http://scdb.wustl.edu/about.php). 

Data source:
- [Download page](http://scdb.wustl.edu/data.php)
- Release ID: 	SCDB_2025_01
- Release Date: 	September 01, 2025
- Includes Terms: 	1946 - 2024

Strategy:
- filter to votes involving current judges
- use `majority` column to generate similarities - 1: dissent, 2: concur
- code dissent as -1, agreement as 1, missing values as 0
- apply MDS to feature matrix

In [67]:
# Read in justice-centered data - one row per judge per vote
df = pd.read_csv(
    "http://scdb.wustl.edu/_brickFiles/2025_01/SCDB_2025_01_justiceCentered_Citation.csv.zip"
)
# Convert to datetime format for filtering
df["dateDecision"] = pd.to_datetime(df["dateDecision"])
df.tail()

C:\Users\maggi\AppData\Local\Temp\ipykernel_96248\170139219.py:2: DtypeWarning: Columns (0: dateRearg) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("http://scdb.wustl.edu/_brickFiles/2025_01/SCDB_2025_01_justiceCentered_Citation.csv.zip")


,caseId,docketId,caseIssuesId,voteId,dateDecision,decisionType,usCite,sctCite,ledCite,lexisCite,...,majVotes,minVotes,justice,justiceName,vote,opinion,direction,majority,firstAgreement,secondAgreement
83639,2024-068,2024-068-01,2024-068-01-01,2024-068-01-01-01-05,2025-06-30,2,606 U.S. 942,145 S. Ct. 2613,222 L. Ed. 2d 1006,2025 U.S. LEXIS 2572,...,9,0,114,EKagan,1.0,1.0,1.0,2.0,0.0,0.0
83640,2024-068,2024-068-01,2024-068-01-01,2024-068-01-01-01-06,2025-06-30,2,606 U.S. 942,145 S. Ct. 2613,222 L. Ed. 2d 1006,2025 U.S. LEXIS 2572,...,9,0,115,NMGorsuch,1.0,1.0,1.0,2.0,0.0,0.0
83641,2024-068,2024-068-01,2024-068-01-01,2024-068-01-01-01-07,2025-06-30,2,606 U.S. 942,145 S. Ct. 2613,222 L. Ed. 2d 1006,2025 U.S. LEXIS 2572,...,9,0,116,BMKavanaugh,1.0,1.0,1.0,2.0,0.0,0.0
83642,2024-068,2024-068-01,2024-068-01-01,2024-068-01-01-01-08,2025-06-30,2,606 U.S. 942,145 S. Ct. 2613,222 L. Ed. 2d 1006,2025 U.S. LEXIS 2572,...,9,0,117,ACBarrett,1.0,1.0,1.0,2.0,0.0,0.0
83643,2024-068,2024-068-01,2024-068-01-01,2024-068-01-01-01-09,2025-06-30,2,606 U.S. 942,145 S. Ct. 2613,222 L. Ed. 2d 1006,2025 U.S. LEXIS 2572,...,9,0,118,KBJackson,1.0,1.0,1.0,2.0,0.0,0.0


Some interesting columns (see codes below)
- vote
- opinion
- majority
- direction

In [ ]:
# df.columns

In [ ]:
opinion_code = {
    1: "justice wrote no opinion",
    2: "justice wrote an opinion",
    3: "justice co-authored an opinion",
}
direction_code = {1: "conservative", 2: "liberal"}
majority_code = {1: "dissent", 2: "majority"}
vote_code = {
    1: "voted with majority or plurality",
    2: "dissent",
    3: "regular concurrence",
    4: "special concurrence",
    5: "judgment of the Court",
    6: "dissent from a denial or dismissal of certiorari , or "
    "dissent from summary affirmation of an appeal",
    7: "jurisdictional dissent",
    8: "justice participated in an equally divided vote ",
}

Preliminary exploration: plot all current and former SC justices by conservative/liberal voting history (using coding from source) and concur/dissent percentage.

For our data: we don't have explicit conservative/liberal codings but could use our issue indicators or a combination of them.

In [ ]:
current_sc_justices = [
    "KBJackson",
    "ACBarrett",
    "BMKavanaugh",
    "NMGorsuch",
    "EKagan",
    "SSotomayor",
    "SAAlito",
    "CThomas",
    "JGRoberts",
]

In [ ]:
# data for viz:
# average direction and majority by justice
data = (
    df.groupby("justiceName")
    .agg({"direction": "mean", "majority": "mean", "dateDecision": "max"})
    .reset_index()
)

data["is_current"] = np.where(data["justiceName"].isin(current_sc_justices), True, False)
data["direction"] = data["direction"] - 1  # adjust to 0-1 scale
data["majority"] = data["majority"] - 1  # adjust to 0-1 scale

In [ ]:
alt.Chart(data, title="Voting History of Supreme Court Justices").mark_point().encode(
    x=alt.X("direction", title="Pct Liberal Vote").scale(zero=False),
    y=alt.Y("majority", title="Pct Concur with majority").scale(zero=False),
    color=alt.Color(
        "is_current",
        legend=alt.Legend(title=None, labelExpr="datum.label == 'true' ? 'Current' : 'Former'"),
    ),
    tooltip=["justiceName"],
).interactive()

#### MDS Procedure

In [ ]:
# restrict to votes of current SC justices
similarity_df = df[df["justiceName"].isin(current_sc_justices)][
    ["docketId", "justiceName", "majority"]
]
similarity_df.dropna(inplace=True)
similarity_df.head()

In [ ]:
similarity_df["similarity_measure"] = similarity_df["majority"].apply(
    lambda x: -1 if x == 1 else (1 if x == 2 else 0)
)
similarity_df.head()

In [ ]:
similarity_wide_df = similarity_df.pivot(
    index="docketId", columns="justiceName", values="similarity_measure"
)
# drop cases without all justices
similarity_wide_df.dropna(inplace=True)
similarity_wide_df = similarity_wide_df.transpose()
similarity_wide_df

In [ ]:
# create context df to join for viz
judge_context = pd.DataFrame(similarity_wide_df.index)
judge_context["appointer"] = [
    "DTrump",
    "DTrump",
    "GHWBush",
    "BObama",
    "GWBush",
    "JBiden",
    "DTrump",
    "GWBush",
    "BObama",
]
judge_context["appointerParty"] = judge_context["appointer"].apply(
    lambda s: "R" if s in ["GWBush", "GHWBush", "DTrump"] else "D"
)
judge_context

In [ ]:
embedding = MDS(n_components=2, metric_mds=True, n_init=1, init="random")
X_transformed = embedding.fit_transform(similarity_wide_df)
x, y = X_transformed.transpose()
judges_mds = judge_context.copy().assign(x=x, y=y)
judges_mds

In [ ]:
alt.Chart(judges_mds, title="SC Judge Similarity").mark_circle(size=60).encode(
    x=alt.X("x", axis=None),
    y=alt.Y("y", axis=None),
    color="appointerParty",
    tooltip=["justiceName", "appointer"],
).interactive()

### K-Means Clustering
Ran k-means clustering on the output of MDS. TODO: run it on original distance measures!

In [ ]:
# run k-means for k=1,2,3,4 and save center points and predicted classifications for viz

center_dfs = dict()
predict = dict()
for n in range(1, 5):
    kmeans = KMeans(n_clusters=n, random_state=0, n_init="auto").fit(judges_mds[["x", "y"]])
    center_dfs[n] = pd.DataFrame(kmeans.cluster_centers_, columns=["x", "y"])
    predict[n] = kmeans.fit_predict(judges_mds[["x", "y"]])

In [ ]:
predict_judges = judges_mds.copy()
for n, prediction in predict.items():
    predict_judges[f"predict{n}"] = prediction

predict_judges

In [ ]:
# set n=1,2,3,4 to see classification with n clusters
n = 3
color_col = f"predict{n}"

chart1 = (
    alt.Chart(predict_judges, title=f"Judge groupings with {n} clusters")
    .mark_point()
    .encode(
        x=alt.X("x", axis=None),
        y=alt.Y("y", axis=None),
        shape=alt.Shape(
            "appointerParty:N",
            legend=alt.Legend(
                title="Appointer Party",
                labelExpr="datum.label == 'D' ? 'Democratic' : 'Republican'",
            ),
        ),
        color=alt.Color(f"predict{n}:N", legend=None),
        tooltip=["justiceName", "appointer"],
    )
    .interactive()
)

chart2 = (
    alt.Chart(center_dfs[n])
    .mark_point(shape="triangle", color="black", size=80)
    .encode(
        x="x",
        y="y",
    )
)

chart = chart1 + chart2
chart